# Time series cross validation: a gap changes the question

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/time-series/time_series_cross_validation.ipynb)

A self-contained CPU notebook. All data are generated locally; no credentials or repository files are needed.

In [ ]:
%pip -q install numpy scipy scikit-learn matplotlib

## Experiment design

The 12-step target is the mean of the next 12 observations. Each input contains the current observation and 23 previous ones. Development targets end strictly before the first future forecast origin. The future evaluation refits once and predicts each origin using observations available at that origin; it does not refit between predictions. Fold-level scaling is fitted only on training rows.

The gap addresses target availability. A later training origin may have a label extending beyond the validation origin even when input rows are ordered. A regime change makes early folds harder than the final future segment, so preserving chronology need not produce an error estimate numerically closer to future error.

In [ ]:
import os
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OPENBLAS_NUM_THREADS"] = "4"
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
import math


In [ ]:
def forecast_data(seed=0):
    rng=np.random.default_rng(seed)
    n=1800; z=np.zeros(n)
    for t in range(1,n):z[t]=.82*z[t-1]+rng.normal(0,.5)
    t=np.arange(n)
    series=2*np.sin(2*np.pi*t/48)+z+np.where(t>=900,3.,0.)
    origins=np.arange(24,n-12)
    X=np.array([series[o-np.arange(24)] for o in origins])
    y=np.array([series[o+1:o+13].mean() for o in origins])
    return series,origins,X,y


In [ ]:
def forecast_experiment():
    records=[]; example=None
    for seed in range(5):
        series,origins,X,y=forecast_data(seed)
        # Every development target is available before the first future origin.
        dev=np.flatnonzero(origins+12<1400); future=np.flatnonzero(origins>=1400)
        methods={'Shuffled':KFold(5,shuffle=True,random_state=42),
                 'Expanding':TimeSeriesSplit(5),
                 'Gap 12':TimeSeriesSplit(5,gap=12)}
        for name,splitter in methods.items():
            scores=[]
            for alpha in [.01,1.,100.,10000.]:
                errors=[]
                for tr,va in splitter.split(dev):
                    assert not np.intersect1d(tr, va).size
                    if name == 'Gap 12':
                        assert origins[dev[tr[-1]]] + 12 < origins[dev[va[0]]]
                    model=make_pipeline(StandardScaler(),Ridge(alpha=alpha))
                    model.fit(X[dev[tr]],y[dev[tr]])
                    errors.append(mean_squared_error(y[dev[va]],model.predict(X[dev[va]])))
                scores.append(np.mean(errors))
            selected=int(np.argmin(scores)); alpha=[.01,1.,100.,10000.][selected]
            model=make_pipeline(StandardScaler(),Ridge(alpha=alpha)).fit(X[dev],y[dev])
            pred=model.predict(X[future]); rmse=mean_squared_error(y[future],pred)**.5
            records.append({'seed':seed,'method':name,'alpha':alpha,'cv_rmse':float(scores[selected]**.5),'future_rmse':float(rmse)})
            if seed==0 and name=='Gap 12': example=(series,origins,X,y,dev,future,pred)
    return records,example


In [ ]:
records, example = forecast_experiment()
for name in ["Shuffled", "Expanding", "Gap 12"]:
    rows = [r for r in records if r["method"] == name]
    print(name, "CV RMSE", np.mean([r["cv_rmse"] for r in rows]),
          "future RMSE", np.mean([r["future_rmse"] for r in rows]),
          "selected alphas", [r["alpha"] for r in rows])
series, origins, X, y, dev, future, pred = example
assert np.max(origins[dev]+12) < np.min(origins[future])
plt.figure(figsize=(9,3)); plt.plot(series); plt.axvline(1400,color="black",linestyle="--")
plt.xlabel("Time index"); plt.ylabel("Value"); plt.title("Synthetic series and future holdout"); plt.show()


## Exercises

1. Change the horizon to six throughout the generator, development boundary and gap. Check label endpoints explicitly.
2. Replace the expanding window with a fixed-length rolling window. Select its length using development data only.
3. Remove the level shift and repeat all five seeds. Keep the final future block untouched until selection is finished.

## Tested environment
Executed on Python 3.12 with NumPy 2.5.3, SciPy 1.18.1, scikit-learn 1.9.1 and Matplotlib 3.11.2. Ranking additionally used XGBoost 3.4.1. Package versions and floating-point libraries can affect timings and fitted results.
